# Atividade de Entrega — Aula 10: pipeline de limpeza de exames laboratoriais

**Análise e Ciência de Dados** (Especialização, IFRN) — Unidade 2, Aula 10

**Nome:** Fábio Noberto Queiroz

**O que entregar (Tarefa no Moodle, prazo 30/10/2026):**
1. este notebook **executado e comentado** (`.ipynb`), com o seu nome no arquivo;
2. o dataset limpo `exames_laboratoriais_limpo.csv`;
3. (opcional) o dicionário de dados em CSV, se você não o deixou no próprio notebook.

**Dados:** `dados/exames_laboratoriais.csv` — 155 registros **sintéticos** de exames laboratoriais (nenhum paciente real).

**Regras de documentação:** antes de cada bloco de código, escreva uma célula de texto explicando **o que** será feito e **por quê**; depois, comente o resultado (o que mudou? quantas linhas/valores foram afetados?). Use os notebooks da aula como referência.

## Requisitos

| # | Etapa | O que fazer |
|---|---|---|
| 1 | Extract e diagnóstico | carregar, `shape`, `dtypes`, ausentes, duplicatas e `value_counts` das categorias |
| 2 | Duplicatas | remover as linhas repetidas e conferir que `id_exame` ficou único |
| 3 | Tipos | `data_coleta` (dois formatos) → `datetime`; `resultado` (`"84,2"`) → `float` |
| 4 | Padronização | `sexo` → `F`/`M`; `prioridade` → `Rotina`/`Urgente`/`Emergência`; `laboratorio` → nomes oficiais sem espaços extras |
| 5 | Valores inválidos e ausentes | o código **999** em `resultado` significa "amostra não processada" → ausente; imputar `resultado` pela mediana **do mesmo tipo de exame** e `idade` pela mediana |
| 6 | Engenharia de atributos | `faixa_etaria` com `pd.cut` (18-39, 40-59, 60-79, 80+); `mes_coleta`; `alterado` (True/False) usando a tabela de referência abaixo |
| 7 | Codificação | `prioridade_cod` ordinal (Rotina=0, Urgente=1, Emergência=2); one-hot de `tipo_exame` |
| 8 | Validação | pelo menos 3 `assert` (ex.: sem ausentes, `id_exame` único, resultados positivos) |
| 9 | Load | exportar `exames_laboratoriais_limpo.csv` (sem índice) e mostrar o **dicionário de dados** (coluna, tipo, descrição) |
| 10 | Conclusão | 3 a 5 linhas: problemas encontrados, decisões tomadas e um achado com os dados limpos |

**Valores de referência (simplificados, apenas para fins didáticos):**

| Exame | Unidade | Faixa considerada normal |
|---|---|---|
| Glicemia | mg/dL | 70 a 99 |
| Colesterol total | mg/dL | abaixo de 190 |
| Creatinina | mg/dL | 0,6 a 1,2 |
| Hemoglobina | g/dL | 12,0 a 17,5 |

Nomes oficiais dos laboratórios: `LabNatal`, `LabSaúde RN`, `Laboratório Central`.

**Pontos de controle (para você conferir):** 150 linhas após remover duplicatas · 4 resultados com código 999 e 6 idades ausentes (após remover duplicatas) · nenhum ausente no dataset final · seguindo exatamente os requisitos, o arquivo final tem 150 linhas e 18 colunas.

**Critérios de avaliação (10 pontos):** limpeza e tipos corretos (3,0) · engenharia de atributos e codificação (2,5) · validação e exportação do dataset (1,5) · documentação — células de texto, comentários e dicionário de dados (2,0) · conclusão (1,0).

## 1. Extract e diagnóstico

_Explique aqui o que vai verificar._  : verificar o total de linhas e colunas do dataset utilizando Shape, verificação de dados duplicados utilizando o comando duplicated, e utilizando value_counts para somar quais são os valores que estão duplicados no dataset.

In [ ]:
import pandas as pd
import numpy as np

ex = pd.read_csv("dados/exames_laboratoriais.csv", decimal=',')
# seu código aqui
print("linhas e colunas:",ex.shape)
print("soma de linhas duplicadas",ex.duplicated().sum())
print("total de duplicados pelo id",ex['id_exame'].value_counts())


ex.head()

linhas e colunas: (155, 10)
soma de linhas duplicadas 5
total de duplicados pelo id id_exame
EX-0091    2
EX-0093    2
EX-0088    2
EX-0086    2
EX-0023    2
          ..
EX-0077    1
EX-0014    1
EX-0082    1
EX-0092    1
EX-0081    1
Name: count, Length: 150, dtype: int64


## 2. Duplicatas

_Explique e comente o resultado._ : utilização do comando drop_duplicates para excluir valores duplicados do dataset, o resultado é a exclusão de 5 registros repetidos do dataset, agora com `150 linhas` e `10 colunas`

In [ ]:
# seu código aqui
ex = ex.drop_duplicates()
print("linhas e colunas:",ex.shape)
print("soma de linhas duplicadas",ex.duplicated().sum())
print("total de duplicados pelo id",ex['id_exame'].value_counts())



linhas e colunas: (150, 10)
soma de linhas duplicadas 0
total de duplicados pelo id id_exame
EX-0120    1
EX-0138    1
EX-0127    1
EX-0066    1
EX-0089    1
          ..
EX-0077    1
EX-0014    1
EX-0082    1
EX-0092    1
EX-0081    1
Name: count, Length: 150, dtype: int64


## 3. Transformação de tipos

_Explique e comente o resultado._ : a conversão da coluna resultado foi realizada na leitura do dataset utilizando o parametro `decimal = ',' ` então o pandas identifica quais coluna numericas possuem a virgula e convertem em numeros reais `float`, ja a coluna `data_coleta` foi utilizada a função astype para converter a coluna em datetime.

In [58]:
# seu código aqui
ex[['data_coleta','resultado']].info()

ex['data_coleta'] = ex['data_coleta'].astype('datetime64[ns]')

ex[['data_coleta','resultado']].info()




<class 'pandas.core.frame.DataFrame'>
Index: 150 entries, 0 to 153
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   data_coleta  150 non-null    object 
 1   resultado    150 non-null    float64
dtypes: float64(1), object(1)
memory usage: 3.5+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 150 entries, 0 to 153
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   data_coleta  150 non-null    datetime64[ns]
 1   resultado    150 non-null    float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 3.5 KB


## 4. Padronização das categorias

_Explique e comente o resultado. : utilização dos comandos `str.strip` para retirar espaços em brancos e a utilização do comando `replace` para padronizar os dados da coluna Sexo e na coluna prioridade apenas inserir o comando `upper` para deixar todos os resultados da coluna em maiusculo.

In [59]:
# seu código aqui

#vericação de padrão preenchido
print("total de grupos",ex['sexo'].value_counts())
print("total de grupors prioridade",ex['prioridade'].value_counts())


#aplicação da padronização
ex['sexo'] = (ex['sexo'].str.strip().str.title().replace({"Masculino":"M","Feminino":"F"}))
ex['prioridade'] = (ex['prioridade'].str.strip().str.upper())


#pós padronização
print("total de grupos pós padronização:",ex['sexo'].value_counts())
print("total de grupors prioridade pós padronização:",ex['prioridade'].value_counts())

 


total de grupos sexo
M            70
F            66
masculino    10
Feminino      4
Name: count, dtype: int64
total de grupors prioridade prioridade
Rotina        95
Urgente       34
ROTINA        10
Emergência     9
URGENTE        1
EMERGÊNCIA     1
Name: count, dtype: int64
total de grupos pós padronização: sexo
M    80
F    70
Name: count, dtype: int64
total de grupors prioridade pós padronização: prioridade
ROTINA        105
URGENTE        35
EMERGÊNCIA     10
Name: count, dtype: int64


## 5. Valores inválidos e ausentes

_Explique e comente o resultado._ : utilização do comando loc para identificar valores = 999 e trocar pela mediana e a realização do comando loc para inserir a mediana em valores NaN na coluna idade.

In [60]:
# seu código aqui
ex.info()
ex['resultado'].value_counts(ascending=False)
ex.loc[ex['resultado'] == 999, 'resultado'] = ex['resultado'].median()

ex['idade'] = ex['idade'].astype(float)
ex.loc[ex['idade'].isna(), 'idade'] = ex['idade'].median()
ex['idade'].isna().sort_values(ascending=True).value_counts()

<class 'pandas.core.frame.DataFrame'>
Index: 150 entries, 0 to 153
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   id_exame     150 non-null    object        
 1   id_paciente  150 non-null    object        
 2   data_coleta  150 non-null    datetime64[ns]
 3   sexo         150 non-null    object        
 4   idade        144 non-null    object        
 5   tipo_exame   150 non-null    object        
 6   resultado    150 non-null    float64       
 7   unidade      150 non-null    object        
 8   laboratorio  150 non-null    object        
 9   prioridade   150 non-null    object        
dtypes: datetime64[ns](1), float64(1), object(8)
memory usage: 12.9+ KB


idade
False    150
Name: count, dtype: int64

## 6. Engenharia de atributos

_Explique e comente o resultado._ 

utilização do comando dt.month para pegar o mes da coleta de acordo com a data da coleta. 

utilizacao do comando pd.cut para separar as faixas de idade

para pecriação do dicionario regra_faixa_exame com 4 faixas e 3 rotulos pois alguns resultados de exames abaixo do padrao e acima do padrão são resultados anormais, utilizando o parametro ordered: False para para poder repetir o valor da label True para valores abaixo e acima do esperado. e a criação da função checar_alteracao para associar em True ou False se os exames estão alterados ou nao


In [ ]:
# seu código aqui

ex['mes_coleta'] = ex['data_coleta'].dt.month

ex['faixa_etaria'] = pd.cut(ex["idade"], bins=[18, 40, 60, 80, 120],
labels=["18-39", "40-59", "60-79", "80+"], right=False)


regra_faixa_exame = {
    'Glicemia':{
        'bins':[0,70,99,np.inf],
        'labels':['True','False','True'],
        'ordered': False
    },
    'Colesterol total':{
            'bins':[0,190,np.inf],
            'labels':['False','True'],
            'ordered': False
        },
    'Creatinina':{
                'bins':[0,0.6,1.2,np.inf],
                'labels':['True','False','True'],
                'ordered': False
            },
    
    'Hemoglobina':{
            'bins':[0,12,17.5,np.inf],
            'labels':['True','False','True'],
            'ordered': False
    }
    
}

def checar_alteracao(grupo):
    nome_exame = grupo.name
    
    if nome_exame in regra_faixa_exame:
        regra = regra_faixa_exame[nome_exame]
        return pd.cut(grupo, bins=regra['bins'], labels=regra['labels'],ordered=False)
    else:
        return np.nan

ex['alterado']  = ex.groupby('tipo_exame')['resultado'].transform(checar_alteracao)


ex['mes_coleta','faixa_etaria','alterado'].head()

,id_exame,id_paciente,data_coleta,sexo,idade,tipo_exame,resultado,unidade,laboratorio,prioridade,prioridade_cod,mes_coleta,faixa_etaria,alterado
0,EX-0120,PAC-0166,2026-08-14,F,73.0,Colesterol total,227.20,mg/dL,Laboratório Central,ROTINA,0,8,60-79,True
1,EX-0138,PAC-0035,2026-08-02,M,19.0,Hemoglobina,12.50,g/dL,Laboratório Central,ROTINA,0,8,18-39,False
2,EX-0127,PAC-0165,2026-08-07,F,60.0,Glicemia,84.20,mg/dL,Laboratório Central,ROTINA,0,8,60-79,False
3,EX-0066,PAC-0043,2026-09-08,M,31.0,Creatinina,1.08,mg/dL,LabSaúde RN,ROTINA,0,9,18-39,False
4,EX-0089,PAC-0092,2026-09-06,M,29.0,Creatinina,0.64,mg/dL,LabNatal,URGENTE,1,9,18-39,False


## 7. Codificação das categorias

_Explique e comente o resultado._

criação do dicionario ordem_prioridade = {"ROTINA": 0, "URGENTE": 1, "EMERGÊNCIA": 2} para associar a prioridade 

e a utilizacao do get_dummies para criar o One-hot encoding e criar colunas de acordo com a classificação do tipo de exame.


In [ ]:
ordem_prioridade = {"ROTINA": 0, "URGENTE": 1, "EMERGÊNCIA": 2}
ex["prioridade_cod"] = ex["prioridade"].map(ordem_prioridade)

print(ex[["prioridade", "prioridade_cod",]].drop_duplicates("prioridade"))


#### ONE-HOT
onehot = pd.get_dummies(ex["tipo_exame"], prefix="t_exame", dtype=int)
pd.concat([ex[["id_exame", "tipo_exame"]], onehot], axis=1).head(5)


   prioridade  prioridade_cod
0      ROTINA               0
4     URGENTE               1
5  EMERGÊNCIA               2


,id_exame,tipo_exame,diag_Colesterol total,diag_Creatinina,diag_Glicemia,diag_Hemoglobina
0,EX-0120,Colesterol total,1,0,0,0
1,EX-0138,Hemoglobina,0,0,0,1
2,EX-0127,Glicemia,0,0,1,0
3,EX-0066,Creatinina,0,1,0,0
4,EX-0089,Creatinina,0,1,0,0


## 8. Validação

_Explique as regras escolhidas._

Utilização do comando Assert para verificar se as colunas informadas estão recebendo os devidos valores, foram testadas as colunas sexo,id_exame e prioridade, todas foram validadas.

In [73]:
# seu código aqui
#assert ex.types.is_float_dtype(ex['resultado'])
assert ex["sexo"].isin(["F", "M"]).all()
assert ex["id_exame"].is_unique
assert ex["prioridade"].isin(["ROTINA", "URGENTE", "EMERGÊNCIA"]).all()
print("Todas as regras de validação passaram ✔")

Todas as regras de validação passaram ✔


## 9. Load e dicionário de dados

_Explique e comente o resultado._

exportação do dataset para csv e criação do dicionario de dados. 

In [85]:
# seu código aqui

limpo = pd.concat([ex, pd.get_dummies(ex["tipo_exame"], prefix="t_exame", dtype=int)], axis=1)

limpo.to_csv('dados/exames_laboritoriais_limpos.csv',index=False,encoding="utf-8")
print(limpo.shape)
print(limpo.head(3).T.to_string())




descricao = {
"id_exame":"identificação do exame",
"id_paciente":"identificação do paciente",
"data_coleta":"data referente ao exame",
"sexo":"sexo do paciente F/M (padronizado)",
"idade":"idade do paciente",
"tipo_exame":"tipo do exame realizado (Creatinina,Glicemia, colesterol , hemoglobina etc)",
"resultado":"valor do resultado do exame",
"unidade":"unidade de medida relacionada ao resultado do tipo de exame realizado",
"laboratorio":"nome do laboratorio onde foi realizado o exame",
"prioridade": "informação se o exame é de rotina, urgencia ou emergência",
"prioridade_cod": "código relacionado a prioridade Rotina = 0 , Urgendia = 1 e Emergencia = 2",
}
dicionario = pd.DataFrame({"coluna": limpo.columns,
                           "tipo": limpo.dtypes.astype(str).values,
                           "descricao": [descricao.get(c, "one-hot do diagnóstico (0/1)") for c in limpo.columns]})
dicionario.to_csv("dados/dicionario_de_dados.csv", index=False, encoding="utf-8")
print(dicionario.to_string(index=False))




(150, 15)
                                            0                    1                    2
id_exame                              EX-0120              EX-0138              EX-0127
id_paciente                          PAC-0166             PAC-0035             PAC-0165
data_coleta               2026-08-14 00:00:00  2026-08-02 00:00:00  2026-08-07 00:00:00
sexo                                        F                    M                    F
idade                                    73.0                 19.0                 60.0
tipo_exame                   Colesterol total          Hemoglobina             Glicemia
resultado                               227.2                 12.5                 84.2
unidade                                 mg/dL                 g/dL                mg/dL
laboratorio               Laboratório Central  Laboratório Central  Laboratório Central
prioridade                             ROTINA               ROTINA               ROTINA
prioridade_cod        

## 10. Conclusão

_Escreva aqui de 3 a 5 linhas._

O processo de Limpeza dos dados é muito importante para criação de uma base para analise dos dados, desde a verificação de duplicatas, tratamento de valores nulos, até a criação do pipeline de dados, verificaçao de engenharia de features para determinar qual seria uma boa feature para o dataset tratado. e a validação dos dados e a criação do dicionario de dados que serão utilizados para analise e ciência de dados. 